In [1]:
import pandas as pd
from deep_translator import GoogleTranslator

from pathlib import Path
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from transformers import AutoTokenizer, AutoModel

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import joblib

import pandas as pd

START_YEAR = 2019

In [2]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)

In [3]:
class Classifier(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, output_dim)
        )

    def forward(self, x):
        return self.net(x)

In [4]:
def embed(texts, batch_size=32):
    embeddings = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            encoded = tokenizer(batch, padding=True, truncation=True, return_tensors='pt').to(device)
            output = model(**encoded)
            cls_embeddings = output.last_hidden_state[:, 0, :]  # [CLS] token
            embeddings.append(cls_embeddings.cpu())
    return torch.cat(embeddings)

CIHR

In [5]:
cihr_path = "raw_data/CIHR/"
cihr_files = Path(cihr_path).glob("*.csv")

CIHR_DFS = [pd.read_csv(f) for f in cihr_files]
CIHR_DATA = pd.concat(CIHR_DFS, ignore_index=True)

In [6]:
grant_descriptors = [
    "ApplicationTitle_TitreDemande", "PrimaryThemeEN_ThemePrincipalAN", "AllResearchCategoriesEN_TousCategoriesRechercheAN", "ApplicationKeywords_MotsClesDemande"
]

col_names = [
    'Title', 'Main_Discipline', 'Area_of_Research', 'Keywords'
]

CIHR_DATA = CIHR_DATA[grant_descriptors]
CIHR_DATA.columns = col_names

CIHR_DATA.drop_duplicates(inplace=True)
CIHR_DATA["Main_Discipline"].value_counts()

Main_Discipline
Biomedical                                         8989
Clinical                                           3869
Social/Cultural/Environmental/Population Health    3003
Health systems/services                            2830
Not applicable/Specified                            127
Name: count, dtype: int64

Training CIHR Main Discipline

In [7]:
tmp_data = CIHR_DATA.sample(frac=1).reset_index(drop=True) # shuffle

# tmp_data = tmp_data.groupby(["Main_Discipline"]).head(500)
tmp_data = tmp_data[tmp_data["Main_Discipline"] != "Not applicable/Specified"]

tmp_data = tmp_data.dropna(subset=['Main_Discipline'])
tmp_data["Main_Discipline"].value_counts()


Main_Discipline
Biomedical                                         8989
Clinical                                           3869
Social/Cultural/Environmental/Population Health    3003
Health systems/services                            2830
Name: count, dtype: int64

In [8]:
label_mapping = dict(enumerate(tmp_data['Main_Discipline'].astype('category').cat.categories))
tmp_data['Main_Discipline'] = tmp_data['Main_Discipline'].astype('category').cat.codes

X_train_text, X_test_text, y_train, y_test = train_test_split(
    tmp_data['Title'], tmp_data['Main_Discipline'], test_size=0.2, stratify=tmp_data['Main_Discipline'], #random_state=42
)

X_train = embed(X_train_text.astype(str).tolist())
X_test = embed(X_test_text.astype(str).tolist())
y_train = torch.tensor(y_train.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

input_dim = X_train.shape[1]
output_dim = len(label_mapping)

clf_model = Classifier(input_dim, output_dim).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(clf_model.parameters(), lr=1e-4)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [9]:
# Train
for epoch in range(500):
    clf_model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = clf_model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

Epoch 1: Loss = 247.5202
Epoch 2: Loss = 180.8206
Epoch 3: Loss = 164.4662
Epoch 4: Loss = 158.8487
Epoch 5: Loss = 155.6500
Epoch 6: Loss = 153.0078
Epoch 7: Loss = 151.0190
Epoch 8: Loss = 149.5318
Epoch 9: Loss = 148.8537
Epoch 10: Loss = 147.0048
Epoch 11: Loss = 146.1226
Epoch 12: Loss = 145.1915
Epoch 13: Loss = 144.1152
Epoch 14: Loss = 143.5088
Epoch 15: Loss = 142.1831
Epoch 16: Loss = 141.4568
Epoch 17: Loss = 140.5685
Epoch 18: Loss = 140.2810
Epoch 19: Loss = 139.6470
Epoch 20: Loss = 138.3007
Epoch 21: Loss = 137.8799
Epoch 22: Loss = 136.9127
Epoch 23: Loss = 136.0421
Epoch 24: Loss = 135.3592
Epoch 25: Loss = 134.8644
Epoch 26: Loss = 134.0493
Epoch 27: Loss = 133.1730
Epoch 28: Loss = 132.6064
Epoch 29: Loss = 132.1150
Epoch 30: Loss = 131.2424
Epoch 31: Loss = 131.1701
Epoch 32: Loss = 130.0918
Epoch 33: Loss = 129.2540
Epoch 34: Loss = 128.6091
Epoch 35: Loss = 128.0777
Epoch 36: Loss = 126.8201
Epoch 37: Loss = 126.1839
Epoch 38: Loss = 125.5025
Epoch 39: Loss = 124.

In [10]:
clf_model.eval()
with torch.no_grad():
    preds = clf_model(X_test.to(device))
    pred_labels = preds.argmax(dim=1).cpu().numpy()
    true_labels = y_test.numpy()

print(classification_report(true_labels, pred_labels, target_names=list(label_mapping.values()), zero_division=0))

                                                 precision    recall  f1-score   support

                                     Biomedical       0.88      0.90      0.89      1798
                                       Clinical       0.60      0.58      0.59       774
                        Health systems/services       0.62      0.61      0.61       566
Social/Cultural/Environmental/Population Health       0.67      0.65      0.66       601

                                       accuracy                           0.75      3739
                                      macro avg       0.69      0.69      0.69      3739
                                   weighted avg       0.75      0.75      0.75      3739



In [11]:
torch.save(clf_model.state_dict(), "models/CIHR_MD.pt")
joblib.dump(label_mapping, "models/CIHR_MD_label_mapping.pkl")


['models/CIHR_MD_label_mapping.pkl']

NSERC

In [12]:
nserc_path = "raw_data/NSERC/"
nserc_files = Path(nserc_path).glob("*.csv")

NSERC_DFS = [pd.read_csv(f) for f in nserc_files]
NSERC_DATA = pd.concat(NSERC_DFS, ignore_index=True)

In [13]:
grant_descriptors = [
    "ApplicationTitle", "AreaOfApplicationGroupEN", "ResearchSubjectEN", "Keyword"
]

col_names = [
     'Title', 'Main_Discipline', 'Area_of_Research', 'Keywords'
]

NSERC_DATA = NSERC_DATA[grant_descriptors]
NSERC_DATA.columns = col_names

NSERC_DATA.drop_duplicates(inplace=True)

Model for Main Discipline

In [14]:
tmp_data = NSERC_DATA.sample(frac=1).reset_index(drop=True) # shuffle

tmp_data = tmp_data[tmp_data["Main_Discipline"] != "Not available"]

tmp_data["Main_Discipline"].value_counts()

Main_Discipline
Advancement of knowledge                   20177
Manufacturing processes and products        4626
Information and communication services      3936
Environment                                 3927
Energy resources                            3123
Health, education and social services       2937
Transportation systems and services         2077
Agriculture and primary food production     1541
Construction, urban and rural planning      1376
Northern development                        1203
Natural resources (economic aspects)        1106
The socioeconomic objective available        737
Commercial services                          400
Name: count, dtype: int64

In [15]:
label_mapping = dict(enumerate(tmp_data['Main_Discipline'].astype('category').cat.categories))
tmp_data['Main_Discipline'] = tmp_data['Main_Discipline'].astype('category').cat.codes

X_train_text, X_test_text, y_train, y_test = train_test_split(
    tmp_data['Title'], tmp_data['Main_Discipline'], test_size=0.2, stratify=tmp_data['Main_Discipline'], #random_state=42
)

X_train = embed(X_train_text.tolist())
X_test = embed(X_test_text.tolist())
y_train = torch.tensor(y_train.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

input_dim = X_train.shape[1]
output_dim = len(label_mapping)

clf_model = Classifier(input_dim, output_dim).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(clf_model.parameters(), lr=1e-4)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [16]:
# Train
for epoch in range(500):
    clf_model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = clf_model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

Epoch 1: Loss = 1054.2893
Epoch 2: Loss = 808.8739
Epoch 3: Loss = 766.7159
Epoch 4: Loss = 745.9380
Epoch 5: Loss = 734.0393
Epoch 6: Loss = 723.6842
Epoch 7: Loss = 716.3764
Epoch 8: Loss = 709.4247
Epoch 9: Loss = 702.9235
Epoch 10: Loss = 696.5361
Epoch 11: Loss = 691.5859
Epoch 12: Loss = 686.3100
Epoch 13: Loss = 680.2272
Epoch 14: Loss = 676.0540
Epoch 15: Loss = 671.7859
Epoch 16: Loss = 666.4977
Epoch 17: Loss = 662.2960
Epoch 18: Loss = 657.5107
Epoch 19: Loss = 654.9985
Epoch 20: Loss = 649.3084
Epoch 21: Loss = 647.1618
Epoch 22: Loss = 640.8138
Epoch 23: Loss = 636.7963
Epoch 24: Loss = 633.9963
Epoch 25: Loss = 631.1024
Epoch 26: Loss = 624.9449
Epoch 27: Loss = 622.5033
Epoch 28: Loss = 617.4083
Epoch 29: Loss = 614.8207
Epoch 30: Loss = 611.4910
Epoch 31: Loss = 607.4561
Epoch 32: Loss = 603.9197
Epoch 33: Loss = 598.4846
Epoch 34: Loss = 597.1486
Epoch 35: Loss = 591.9071
Epoch 36: Loss = 587.9329
Epoch 37: Loss = 586.7528
Epoch 38: Loss = 583.0121
Epoch 39: Loss = 579

In [17]:
clf_model.eval()
with torch.no_grad():
    preds = clf_model(X_test.to(device))
    pred_labels = preds.argmax(dim=1).cpu().numpy()
    true_labels = y_test.numpy()

print(classification_report(true_labels, pred_labels, target_names=list(label_mapping.values()), zero_division=0))

                                         precision    recall  f1-score   support

               Advancement of knowledge       0.81      0.90      0.86      4036
Agriculture and primary food production       0.84      0.73      0.78       308
                    Commercial services       0.65      0.50      0.56        80
 Construction, urban and rural planning       0.84      0.78      0.81       275
                       Energy resources       0.82      0.79      0.80       625
                            Environment       0.78      0.76      0.77       786
  Health, education and social services       0.77      0.71      0.74       588
 Information and communication services       0.85      0.80      0.82       787
   Manufacturing processes and products       0.72      0.67      0.70       925
   Natural resources (economic aspects)       0.78      0.63      0.70       221
                   Northern development       0.77      0.61      0.68       241
  The socioeconomic objecti

In [18]:
torch.save(clf_model.state_dict(), "models/NSERC_MD.pt")
joblib.dump(label_mapping, "models/NSERC_MD_label_mapping.pkl")

['models/NSERC_MD_label_mapping.pkl']

SSHRC

In [19]:
sshrc_path = "raw_data/SSHRC/"
sshrc_files = Path(sshrc_path).glob("*.csv")

SSHRC_DFS = [pd.read_csv(f) for f in sshrc_files]
SSHRC_DATA = pd.concat(SSHRC_DFS, ignore_index=True)

In [20]:
grant_descriptors = [
    "Title-Titre", "Main_Discipline", "Area_of_Research", "Keywords-Mots-clés"
]

col_names = [
    'Title', 'Main_Discipline', 'Area_of_Research', 'Keywords'
]


SSHRC_DATA = SSHRC_DATA[grant_descriptors]
SSHRC_DATA.columns = col_names

SSHRC_DATA.drop_duplicates(inplace=True)

In [21]:
SSHRC_DATA["Main_Discipline"].value_counts()

tmp_data = SSHRC_DATA.sample(frac=1).reset_index(drop=True) # shuffle

val_counts = tmp_data["Main_Discipline"].value_counts()
classes = val_counts[val_counts > 200].index
tmp_data = tmp_data[tmp_data["Main_Discipline"].isin(classes)]

tmp_data = tmp_data.groupby(["Main_Discipline"]).head(500)
tmp_data = tmp_data.dropna(subset=['Main_Discipline'])
tmp_data = tmp_data[~(tmp_data['Main_Discipline'].isin(["Not Specified", "Not specified", "Not Applicable", "Multiple primary fields of research", "Interdisciplinary Studies"]))]


tmp_data["Main_Discipline"].value_counts()


Main_Discipline
Psychology                                           500
Geography                                            500
Management, Business, Administrative Studies         500
Anthropology                                         500
Law                                                  500
Communications and Media Studies                     500
Social Work                                          500
History                                              500
Philosophy                                           500
Economics                                            500
Fine Arts                                            500
Urban and Regional Studies, Environmental Studies    500
Linguistics                                          500
Political Science                                    500
Sociology                                            500
Literature, Modern Languages and                     500
Education                                            500
Criminology    

In [22]:
label_mapping = dict(enumerate(tmp_data['Main_Discipline'].astype('category').cat.categories))
tmp_data['Main_Discipline'] = tmp_data['Main_Discipline'].astype('category').cat.codes

X_train_text, X_test_text, y_train, y_test = train_test_split(
    tmp_data['Title'], tmp_data['Main_Discipline'], test_size=0.2, stratify=tmp_data['Main_Discipline'], #random_state=42
)

X_train = embed(X_train_text.astype(str).tolist())
X_test = embed(X_test_text.astype(str).tolist())
y_train = torch.tensor(y_train.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

input_dim = X_train.shape[1]
output_dim = len(label_mapping)

clf_model = Classifier(input_dim, output_dim).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(clf_model.parameters(), lr=1e-4)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [23]:
# Train
for epoch in range(500):
    clf_model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = clf_model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

Epoch 1: Loss = 360.4394
Epoch 2: Loss = 343.7647
Epoch 3: Loss = 318.8702
Epoch 4: Loss = 292.0480
Epoch 5: Loss = 269.7406
Epoch 6: Loss = 253.1226
Epoch 7: Loss = 241.3079
Epoch 8: Loss = 233.5151
Epoch 9: Loss = 226.0867
Epoch 10: Loss = 220.6566
Epoch 11: Loss = 216.4578
Epoch 12: Loss = 212.8838
Epoch 13: Loss = 209.7242
Epoch 14: Loss = 207.4318
Epoch 15: Loss = 204.4872
Epoch 16: Loss = 202.8379
Epoch 17: Loss = 200.9718
Epoch 18: Loss = 198.9915
Epoch 19: Loss = 198.1226
Epoch 20: Loss = 196.4833
Epoch 21: Loss = 194.6256
Epoch 22: Loss = 193.1318
Epoch 23: Loss = 192.0648
Epoch 24: Loss = 190.7791
Epoch 25: Loss = 190.2694
Epoch 26: Loss = 188.7820
Epoch 27: Loss = 188.2013
Epoch 28: Loss = 187.1392
Epoch 29: Loss = 186.4775
Epoch 30: Loss = 184.7263
Epoch 31: Loss = 184.1428
Epoch 32: Loss = 183.4951
Epoch 33: Loss = 182.7823
Epoch 34: Loss = 181.3333
Epoch 35: Loss = 181.3670
Epoch 36: Loss = 179.9742
Epoch 37: Loss = 178.8475
Epoch 38: Loss = 178.2883
Epoch 39: Loss = 177.

In [24]:
clf_model.eval()
with torch.no_grad():
    preds = clf_model(X_test.to(device))
    pred_labels = preds.argmax(dim=1).cpu().numpy()
    true_labels = y_test.numpy()
    
print(classification_report(true_labels, pred_labels, target_names=list(label_mapping.values()), zero_division=0))

                                                   precision    recall  f1-score   support

                                     Anthropology       0.31      0.30      0.30       100
                                      Archaeology       0.74      0.77      0.75        81
                 Communications and Media Studies       0.37      0.34      0.35       100
                                      Criminology       0.53      0.53      0.53        93
                                        Economics       0.59      0.51      0.55       100
                                        Education       0.60      0.62      0.61       100
                                        Fine Arts       0.45      0.45      0.45       100
                                        Geography       0.31      0.28      0.30       100
                                          History       0.50      0.43      0.46       100
                                              Law       0.57      0.52      0.54       10

In [25]:
torch.save(clf_model.state_dict(), "models/SSHRC_MD.pt")
joblib.dump(label_mapping, "models/SSHRC_MD_label_mapping.pkl")

['models/SSHRC_MD_label_mapping.pkl']